# Calibration X6Y3 — the qcal chain on the real chip (ZCU216)

The hardware notebook `Calibration_X6Y3.ipynb` (qcal + QubiC), reproduced step-for-step with
`riscq.cal` on a **real ZCU216** driving the X6Y3 chip (spec
[13-qcal-parity](../specs/software/13-qcal-parity.md)). The config of record is the real X6Y3
**qcal tree** ([`cal-config-x6y3.yaml`](cal-config-x6y3.yaml)): 8 qubits, FAST_DRAG X90s,
readout at 6.55–6.84 GHz, `readout/herald: true`, 500 µs passive reset — loaded with
`Config.from_qcal`, calibrated fields written back with `save_qcal`.

Like the other hardware notebooks (`remote_pulse`, `vna`, `iq_scatter`) this connects to the
board server over `RemoteDriver` and is **not executed in CI** — its co-sim twin
([`calibration_x6y3_cosim.ipynb`](calibration_x6y3_cosim.ipynb)) runs the same chain against a
planted `TwoLevelModel` and is the verified reference for every code path used here.

Step-for-step vs the qcal reference (same knobs, same qubit subsets):

| reference cell | here |
|---|---|
| `ReadoutCalibration(qubits=[0..7], gate='X90')` | same — fixes each `demod/phase` + `res_sign` (the on-chip discriminator) |
| `Separation`: `readout/{q}/freq` ± 2.5 MHz, 31 pts | same span/points; argmax of the two-state **cluster SNR**; RAW shots RAM-sized (31 × 32 × 2 ≈ 8 KB of 16 KB, spec 13 §5) |
| `Fidelity`: `readout/{q}/amp` ± 0.005, 31 pts, 3000 shots | same — scored by the `res` bit under the FIXED discriminator |
| `ReadoutFidelity`, 5000 shots | same — heralded confusion matrix |
| `Frequency(qubits=(2,5), detunings=±2.5,±5 MHz, t_max=1 µs, 512 shots)` | same — V-fit `a·|x−b|+c` |
| `Amplitude(qubits=[2,5])` coarse `n_gates=1` + fine `relative_amp` `n_gates=4` | same two passes |
| `Phase` relative loop over all 8, then absolute on `[1,3,5,7]` | same — `Y180_X90`/`X180_Y90` line crossing on the virtual-Z pair |

Deliberate differences (spec 13 §2): sweeps are **on-core loops**, discrimination is the **on-chip
`res` bit**, and a failed fit **refuses** to update the config. Every cell reloads the working tree
(the reference's `cfg.load()`); applied proposals are saved back — the full qcal round trip.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

import riscq
from riscq.cal import (Config, ReadoutCalibration, Separation, Fidelity, ReadoutFidelity,
                       Frequency, Amplitude, Phase)
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.map import SocMap, SocParams
from riscq.pulses import units

MHz, us = 1e6, 1e-6

## Connect to the board and the config of record

The reference preamble points at a chassis (`ip_address`/`port`) and a config dir. Here the chassis
is the ZCU216 board server ([docs/software/board-server.md](../docs/software/board-server.md)) with
an X6Y3 gateware bundle loaded, and the config is the real qcal tree, copied to a **working file**
that every cell reloads and every applied proposal is saved back into.

In [ ]:
BOARD = '192.168.1.122'                   # the ZCU216's LAN address (or the full PYRO: uri)
PORT = 9091

SW = Path(riscq.__file__).resolve().parents[1]              # .../software
SRC = SW.parent / 'examples' / 'cal-config-x6y3.yaml'       # the real X6Y3 qcal tree
WORK = SW / 'build' / 'x6y3_config.yaml'                    # the working copy (the reference's basedir)
WORK.parent.mkdir(exist_ok=True)
shutil.copy(SRC, WORK)

drv = RemoteDriver(BOARD, PORT)
print('server:', drv.board.info())

# first time only — push the X6Y3 build up and load it:
# upload_bundle(drv, 'x6y3', xsa='../build/x6y3/top.xsa',
#               params_json='../software/configs/x6y3.json')
# info = drv.board.load('x6y3')
# assert info['mts_result'] == 0, 'multi-tile sync missed its target latencies'

m = SocMap(SocParams.from_json(drv.board.get_params()))
QUBITS = list(range(8))

cfg = Config.from_qcal(WORK)
cfg.check_hardware(m.params)     # the tree's DAC/ADC rates + interpolation must describe THIS bundle
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores, "
      f"{units.sample_rate(m.params) / 1e9:.0f} GS/s DACs;  herald = {cfg['readout/herald']}")

def step(cal):
    '''run -> print -> apply if ok -> persist to the working tree (qcal's auto write-back).'''
    r = cal.run(drv)
    print(f'{r.label}: ok={r.ok}  proposal={r.proposal}')
    if r.ok:
        r.apply()
        cfg.save_qcal(WORK)
    else:
        print('  fit failed — config left unchanged (fail-loud, spec 13 §2)')
    return r

# Readout Calibration

Reference: `ReadoutCalibration(..., gate='X90', n_shots=5000, n_levels=2)` — prep |0⟩ and |1⟩
(X90·X90) on all 8 qubits simultaneously, cluster the raw IQ, persist the classifier. The proposal
fixes each qubit's `demod/phase` so the |0⟩→|1⟩ cluster axis lands on +real, where the on-chip
`sign(sumR)` discriminates (spec 13 Q2). Raw-IQ shots are captured in one rerun per prep state, so
they are core-RAM-bounded: 1000 shots (8 KB of 16 KB) instead of the reference's host-streamed 5000.

In [ ]:
cfg = Config.from_qcal(WORK)
rc = step(ReadoutCalibration(cfg, QUBITS, shots=1000, gate='X90'))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for q, ax in zip(QUBITS, axes.flat):
    d = rc.data[q]
    ax.scatter(d['iq0'][:, 0], d['iq0'][:, 1], s=4, label='|0>')
    ax.scatter(d['iq1'][:, 0], d['iq1'][:, 1], s=4, label='|1>')
    ax.set_title(f"q{q}  sep={d['separation']:.2f}")
    ax.set_aspect('equal'); ax.legend()
plt.tight_layout(); plt.show()

## Separation

Reference: sweep `readout/{q}/freq` ± 2.5 MHz (31 points) around each qubit's current value, both
prep states, argmax of the cluster SNR (`‖Δmeans‖ / Σ(2√cov)`). The matched DAC+demod pair is swept
on-core in RAW mode; 32 shots/point/prep is the 16 KB-RAM budget (spec 13 §5).

In [ ]:
cfg = Config.from_qcal(WORK)
sep = step(Separation(cfg, QUBITS, span=2.5 * MHz, points=31, shots=32, gate='X90'))

for q in QUBITS:
    plt.plot((sep.data[q]['x'] - sep.data[q]['x'].mean()) / MHz, sep.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout freq − centre [MHz]'); plt.ylabel('cluster SNR'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"readout/{q}/freq -> {cfg[f'readout/{q}/freq'] / 1e9:.6f} GHz")

## Fidelity

Reference: sweep `readout/{q}/amp` ± 0.005 (31 points, 3000 shots) under the **fixed** classifier,
argmax of the confusion diagonal. Same knob and scoring — the discriminator is the `res` bit whose
phase ReadoutCalibration fixed, never retrained per point. (The reference disables ESP here; ESP is
a 3-level non-goal, spec 13 §11 — this chain is 2-level throughout. X6Y3's smallest readout amp is
0.0115, so the ± 0.005 span reaches down to 0.0065 — swept in full, no floor.)

In [ ]:
cfg = Config.from_qcal(WORK)
fid = step(Fidelity(cfg, QUBITS, amp_span=0.005, points=31, shots=3000, gate='X90'))

for q in QUBITS:
    plt.plot(fid.data[q]['x'], fid.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout amp'); plt.ylabel('confusion diagonal'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"readout/{q}/amp -> {cfg[f'readout/{q}/amp']:.5f}")

# Readout Fidelity

Reference: `ReadoutFidelity(..., classifier=classifier, n_shots=5000)` — the confusion matrix under
the classifier passed in, no retraining. Same here: two COUNTS reruns at the calibrated amp,
straight off the `res` bit, heralded (every counts-mode shot on this `herald: true` config is
post-selected on a pre-sequence |0⟩ read, exactly like qcal's transpiler).

In [ ]:
cfg = Config.from_qcal(WORK)
rof = step(ReadoutFidelity(cfg, QUBITS, shots=5000, gate='X90'))

for q in QUBITS:
    print(f"q{q} confusion (row = prepared, col = classified):")
    print(np.round(rof.data[q]['confusion'], 3), f"  fidelity={rof.data[q]['fidelity']:.3f}")

# Single Qubit
## GE
### Freq

Reference: Ramsey vs detuning on qubits (2, 5) — `detunings = [-5, -2.5, 2.5, 5] MHz`,
`t_max = 1 µs`, 512 shots; fit `a·|x − b| + c` over the unsigned fringe frequencies and correct the
carrier by the vertex.

In [ ]:
cfg = Config.from_qcal(WORK)
detunings = np.array([-5, -2.5, 2.5, 5]) * MHz          # the reference's exact set
fr = step(Frequency(cfg, (2, 5), detunings=detunings, t_max=1 * us, points=30, shots=512))

for q in (2, 5):
    d = fr.data[q]
    plt.plot(d['applied'], d['obs'], 'o', label=f'q{q}')
plt.xlabel('applied detuning [codes]'); plt.ylabel('|fringe| [codes]'); plt.legend(); plt.show()
for q in (2, 5):
    print(f"qubit/{q}/freq -> {cfg[f'qubit/{q}/freq'] / 1e9:.6f} GHz")

### Amplitude

Reference (qubits [2, 5]): a coarse `n_gates=1` Rabi over the full amp range (31 points), then the
fine pass — 4 repeated X90s with `relative_amp=True` sweeping 0.7–1.3× the coarse result (each X90
is a quarter period, so the train must be a multiple of 4 — qcal's own guard).

In [ ]:
cfg = Config.from_qcal(WORK)
ac = step(Amplitude(cfg, [2, 5], gate='X90', n_gates=1, amp_span=(0.03, 0.97), points=31, shots=512))

for q in (2, 5):
    plt.plot(ac.data[q]['x'] / units.AMP_SCALE, ac.data[q]['y'], 'o-', label=f'q{q}')
plt.xlabel('X90 amp'); plt.ylabel('P(|1>)'); plt.legend(); plt.show()
for q in (2, 5):
    print(f"q{q}: coarse X90 amp -> {cfg[f'qubit/{q}/x90/amp']:.5f}")

In [ ]:
cfg = Config.from_qcal(WORK)
af = step(Amplitude(cfg, [2, 5], gate='X90', n_gates=4, amp_span=(0.7, 1.3), relative_amp=True,
                    points=31, shots=512))
for q in (2, 5):
    print(f"q{q}: fine X90 amp -> {cfg[f'qubit/{q}/x90/amp']:.5f}")

### Phase

Reference: two passes — a per-qubit loop over all 8 with `relative_phase=True` (± 0.25, 21 points),
then a wider absolute pass on `[1, 3, 5, 7]` (± 0.3, 31 points). The swept knob is the X90's
**virtual-Z pair** (`qubit/{q}/x90/vz` — nonzero and asymmetric on this config, e.g. q6); each point
runs qcal's two sequences (`Y180_X90` / `X180_Y90`) and the calibrated value is the crossing of the
two fitted lines (spec 13 Q3), written to both slots as qcal does.

In [ ]:
cfg = Config.from_qcal(WORK)
for q in QUBITS:                                 # the reference's per-qubit relative loop
    step(Phase(cfg, [q], span=0.25, points=21, shots=512, relative_phase=True))

In [ ]:
cfg = Config.from_qcal(WORK)
ph = step(Phase(cfg, [1, 3, 5, 7], span=0.3, points=31, shots=512))   # the wider absolute pass

for q in QUBITS:
    vz = cfg.get(f'qubit/{q}/x90/vz', [0.0, 0.0])
    print(f'q{q}: X90 virtual-Z pair -> [{vz[0]:+.4f}, {vz[1]:+.4f}] rad')

## Summary — the calibrated tree

Every step ran through the full qcal round trip: `Config.from_qcal` → calibrate → `save_qcal` back
into the working tree. Diff `WORK` against `cal-config-x6y3.yaml` to see exactly what moved.

In [ ]:
cfg = Config.from_qcal(WORK)
for q in QUBITS:
    vz = cfg.get(f'qubit/{q}/x90/vz', [0.0, 0.0])
    print(f"q{q}: f_ge={cfg[f'qubit/{q}/freq'] / 1e9:.6f} GHz  x90 amp={cfg[f'qubit/{q}/x90/amp']:.5f}  "
          f"vz=[{vz[0]:+.4f},{vz[1]:+.4f}]  "
          f"readout {cfg[f'readout/{q}/freq'] / 1e9:.6f} GHz @ {cfg[f'readout/{q}/amp']:.5f}  "
          f"fidelity={rof.data[q]['fidelity']:.3f}")
print(f'\ncalibrated tree: {WORK}')

## Disconnect

In [ ]:
drv.close()
print('disconnected')